In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [2]:
# ============================================================
# LOAD BEST CONFIG
# ============================================================

import json

# Open the file and parse its contents
with open('descriptornet_datasetB_bho/best_config.json', 'r') as file:
    best_configuration = json.load(file)
best_configuration

{'depth': 6,
 'width': 256,
 'activation': 'tanh',
 'lr': 0.0027730839047650853,
 'batch_size': 128,
 'weight_decay': 0.0001}

In [34]:
# ============================================================
# CONFIG
# ============================================================
DATA_DIR     = "../plga_dataset/release_dataset_with_Crank_release_even.xlsx"
NET2_WEIGHTS = "../physicsnet/physicsnet_pretrained.pt"
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# --- Network 1 (trained) ---
N1_DEPTH = best_configuration["depth"]
N1_WIDTH = best_configuration["width"]
N1_ACT   = best_configuration["activation"]


# --- Network 2 (frozen, must match pretrain_net2.py exactly) ---
N2_DEPTH = 3
N2_WIDTH = 256
N2_ACT = "silu"
 
# --- Training ---
WEIGHT_DECAY      = best_configuration['weight_decay']
EPOCHS            = 1000
LR                = best_configuration["lr"]
BATCH_SIZE        = best_configuration["batch_size"]        # rows (particle x timepoint pairs)
N_RHO_INTEGRATION = 30
 
# --- Data ---
DESCRIPTOR_COLS = [
    'Drug MW', 'Drug TPSA', 'Drug LogP', 'Polymer MW', 'LA/GA',
    'Initial Drug-to-Polymer Ratio', 'Particle Size',
    'Drug Loading Capacity', 'Drug Encapsulation Efficiency',
    'Solubility Enhancer Concentration'
]
SEED = 3
torch.manual_seed(SEED)
np.random.seed(SEED)

In [35]:
# ============================================================
# DATA LOADING
# ============================================================
def load_data(path, release_column="Crank_Release"):
    """
    Load flat table. Each row is one (particle, timepoint) pair.
    Returns flat numpy arrays ready for TensorDataset.
    Also returns per-particle arrays for D log-err evaluation.
    """
    df = pd.read_excel(path)
 
    # Unit conversions
    df['t_s']  = df['Time'] * 86400.0          # days -> seconds
    df['R_cm'] = df['Particle Size'] / 2.0 / 1e4  # diameter um -> radius cm
 
    # Flat arrays (one row per particle x timepoint)
    X = df[DESCRIPTOR_COLS].values.astype(float)   # (N_rows, n_desc)
    t = df['t_s'].values.astype(float)              # (N_rows,)
    R = df['R_cm'].values.astype(float)             # (N_rows,)
    c = df[release_column].values.astype(float)    # (N_rows,)
    D = df['Crank_D'].values.astype(float) * 10**4         # (N_rows,) eval only
 
    # Per-particle arrays for D log-err evaluation
    df_part = (df[['Formulation Index', 'Crank_D', 'R_cm'] + DESCRIPTOR_COLS]
               .drop_duplicates(subset=['Formulation Index'])
               .reset_index(drop=True))
    X_part = df_part[DESCRIPTOR_COLS].values.astype(float)
    D_part = df_part['Crank_D'].values.astype(float) * 10**4
    fid    = df_part['Formulation Index'].values
 
    # Fo sanity check
    Fo_approx = D * t / R**2
    print(f"D range: [{D.min():.3e}, {D.max():.3e}] cm^2/s")
    print(f"R range: [{R.min():.3e}, {R.max():.3e}] cm")
    print(f"t range: [{t.min():.1f}, {t.max():.1f}] s")
    print(f"Fo range (approx): [{Fo_approx.min():.4f}, {Fo_approx.max():.4f}]")
    if Fo_approx.max() > 1.5:
        print(f"  WARNING: {(Fo_approx > 1.5).sum()} rows have Fo > 1.5 "
              f"(will be clamped to PhysicsNet ceiling).")
 
    return (X, t, R, c, D,
            X_part, D_part, fid)

In [36]:
# ============================================================
# NETWORKS
# ============================================================
def get_activation(name):
    return {"tanh": nn.Tanh(), "silu": nn.SiLU(), "gelu": nn.GELU()}[name.lower()]
 
 
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, depth, width, activation):
        super().__init__()
        layers = [nn.Linear(in_dim, width), get_activation(activation)]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), get_activation(activation)]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
 
    def forward(self, x):
        return self.net(x)
 
 
class DescriptorNet(nn.Module):
    """Network 1 (trained): descriptors -> log10(D_eff in cm^2/s)."""
    def __init__(self, n_desc, depth=N1_DEPTH, width=N1_WIDTH,
                 activation=N1_ACT, log10_D_init=-16.0):
        super().__init__()
        self.mlp = MLP(n_desc, 1, depth, width, activation)
        nn.init.constant_(self.mlp.net[-1].bias, log10_D_init)
 
    def forward(self, desc):
        return self.mlp(desc).squeeze(-1)      # (B,)
 
 
class PhysicsNet(nn.Module):
    """Network 2 (frozen): (rho, Fo) -> u(rho, Fo)."""
    def __init__(self, depth=N2_DEPTH, width=N2_WIDTH, activation=N2_ACT):
        super().__init__()
        self.mlp = MLP(2, 1, depth, width, activation)
 
    def forward(self, rho, Fo):
        x   = torch.cat([rho**2, Fo], dim=1)
        raw = self.mlp(x)
        return (1.0 - rho**2) * raw            # hard BC: u(rho=1) = 0


In [37]:
# ============================================================
# INTEGRATION
# ============================================================
def integrate_release(net2, Fo, rho_grid):
    """
    Compute fractional release for a batch of Fo values.
 
    Fo       : (B,)
    rho_grid : (n_rho,)
    Returns release : (B,)
    """
    B    = Fo.shape[0]
    n_rho = rho_grid.shape[0]
 
    Fo_exp  = Fo.view(B, 1).expand(B, n_rho).reshape(-1, 1)   # (B*n_rho, 1)
    rho_exp = rho_grid.view(1, n_rho).expand(B, n_rho).reshape(-1, 1)
 
    u   = net2(rho_exp, Fo_exp).view(B, n_rho)                # (B, n_rho)
    M_t = torch.trapz(u * rho_grid**2, rho_grid, dim=1)       # (B,)
 
    return torch.clamp(1.0 - 3.0 * M_t, min=0.0, max=1.0)    # (B,)
 
 
# ============================================================
# D LOG-ERR EVALUATION (per particle, not per row)
# ============================================================
@torch.no_grad()
def eval_D_logerr(net1, X_part_tensor, D_part_tensor):
    """Mean absolute log10 D error across particles."""
    net1.eval()
    log10_D_pred = net1(X_part_tensor)
    D_pred       = 10.0 ** log10_D_pred
    log_err      = torch.mean(
        torch.abs(torch.log10(D_pred) - torch.log10(D_part_tensor))
    )
    return log_err.item()

In [38]:
# ============================================================
# LOAD DATA AND RECREATE THE EXACT SEED PREPROCESSING
# ============================================================

(X, t, R, c, D,
 X_part, D_part, fid) = load_data(
    DATA_DIR,
)

n_desc = X.shape[1]
print(f"\n{n_desc} descriptor columns.\n")

# Formulation index for every row
df_raw = pd.read_excel(DATA_DIR)
row_fids = df_raw["Formulation Index"].values

# Recreate the fixed outer 80/20 formulation split used in training
all_fids = np.unique(fid)
fids_train, fids_test = train_test_split(
    all_fids,
    test_size=0.2,
    random_state=42,
)

internal_val_fraction = 0.15

rng = np.random.default_rng(SEED)
shuffled_train_fids = fids_train.copy()
rng.shuffle(shuffled_train_fids)

n_ival = max(
    1,
    int(len(fids_train) * internal_val_fraction),
)

ival_fids = shuffled_train_fids[:n_ival]
actual_tr_fids = shuffled_train_fids[n_ival:]

# Row-level masks
actual_tr_mask = np.isin(row_fids, actual_tr_fids)
ival_mask = np.isin(row_fids, ival_fids)
train_mask = np.isin(row_fids, fids_train)
test_mask = np.isin(row_fids, fids_test)

scaler = StandardScaler()
scaler.fit(X[actual_tr_mask])

# Scale row-level descriptors without refitting the scaler
X_actual_tr_scaled = scaler.transform(X[actual_tr_mask])
X_ival_scaled = scaler.transform(X[ival_mask])
X_train_scaled = scaler.transform(X[train_mask])
X_test_scaled = scaler.transform(X[test_mask])

# Per-formulation masks
part_train_mask = np.isin(fid, fids_train)
part_test_mask = np.isin(fid, fids_test)

# Per-formulation tensors for log10(D) error evaluation
X_part_train = torch.tensor(
    scaler.transform(X_part[part_train_mask]),
    dtype=torch.float32,
    device=DEVICE,
)

X_part_test = torch.tensor(
    scaler.transform(X_part[part_test_mask]),
    dtype=torch.float32,
    device=DEVICE,
)

D_part_train = torch.tensor(
    D_part[part_train_mask],
    dtype=torch.float32,
    device=DEVICE,
)

D_part_test = torch.tensor(
    D_part[part_test_mask],
    dtype=torch.float32,
    device=DEVICE,
)

print(
    f"Actual training formulations: {len(actual_tr_fids)}\n"
    f"Internal validation formulations: {len(ival_fids)}\n"
    f"Outer test formulations: {len(fids_test)}"
)

D range: [7.181e-17, 2.736e-11] cm^2/s
R range: [5.765e-05, 1.478e-02] cm
t range: [0.0, 20540069.3] s
Fo range (approx): [0.0000, 1.7138]

10 descriptor columns.

Actual training formulations: 218
Internal validation formulations: 38
Outer test formulations: 65


In [39]:
log10_D_init = -10
DescriptorNet_ = DescriptorNet(n_desc=n_desc,
                         log10_D_init=log10_D_init).to(DEVICE)


state_dict = torch.load(
    f"descriptornet_datasetB_bho/best_model/seed_{SEED}/weights.pt",
    map_location=DEVICE,
    weights_only=True,
)


DescriptorNet_.load_state_dict(state_dict)
DescriptorNet_.eval()



DescriptorNet(
  (mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=10, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): Tanh()
      (8): Linear(in_features=256, out_features=256, bias=True)
      (9): Tanh()
      (10): Linear(in_features=256, out_features=256, bias=True)
      (11): Tanh()
      (12): Linear(in_features=256, out_features=1, bias=True)
    )
  )
)

In [40]:
print(f"TRAIN LOG D ERROR : {eval_D_logerr(DescriptorNet_, X_part_train, D_part_train)}")
print(f"TEST  LOG D ERROR : {eval_D_logerr(DescriptorNet_, X_part_test, D_part_test)}")

TRAIN LOG D ERROR : 0.1430014669895172
TEST  LOG D ERROR : 0.42621690034866333
